In [ ]:
# Setup: Clone repo and install dependencies
from IPython.display import clear_output

!git clone --branch shantanu https://github.com/AISC-Linear-Probe-Gen/Probe-Generalisation.git
%pip install --upgrade mech-interp-toolkit

clear_output()

import os
os.chdir("/content/Probe-Generalisation/research/obfuscated_activations")

In [ ]:
# Imports
from datasets import load_dataset
import torch
import einops
from pathlib import Path
import warnings
import json
from tqdm import tqdm

from utils.data import extract_user_instruction
from mech_interp_toolkit.utils import load_model_tokenizer_config, set_global_seed
from mech_interp_toolkit.activation_utils import get_embeddings_dict

warnings.filterwarnings("ignore")

set_global_seed(0)
torch.set_grad_enabled(False)

In [ ]:
# Configuration
dataset_name = "Mechanistic-Anomaly-Detection/llama3-jailbreaks"
split = "circuit_breakers_train"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
suffix_path = "suffix.pt"
batch_size = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load model, tokenizer, and data
model, ch_tokenizer, config = load_model_tokenizer_config(model_name, device=device)

# Load dataset
dataset = load_dataset(dataset_name, split=split)
prompts_str = [extract_user_instruction(item) for item in dataset]

# Load adversarial suffix
suffix_embed = torch.load(suffix_path, map_location=device)
len_suffix = suffix_embed.shape[1]

print(f"Loaded {len(prompts_str)} prompts from dataset")
print(f"Suffix shape: {suffix_embed.shape}")

In [ ]:
# Display sample results
print("Sample generated text with suffix injection:\n")
for idx, result in enumerate(results[:5]):
    print(f"Sample {idx}:")
    print(f"Prompt: {result['prompt'][:100]}...")  # Show first 100 chars
    print(f"Generation: {result['generation']}")
    print("-" * 80)

In [ ]:
# Save generations to JSON file
save_path = Path(f"outputs/generations/{split}_generations_with_suffix.json")
save_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Saving {len(results)} generations to {save_path}")

with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✓ Saved generations to {save_path}")